# RECALL-MM 数据预处理

## 1　读取与概览

运行所有单元即可核对原始召回数据。默认查找同目录或下载目录下的 dataset.json，也可以在 DATA_PATH 指定文件。数据概览记录行数、召回编号数、日期范围、完全重复行和各字段缺失。来源行号从 1 开始，用来回查原始记录。

In [1]:
from hashlib import sha256
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_PATH = None   # 也可填写 Path("/path/to/dataset.json")
EXPORT_DIR = Path("preprocessing_output")  # 设为 None 时仅查看结果
FIRST_YEAR, TRAIN_END, LAST_YEAR = 2000, 2018, 2024

text_fields = ["product_name", "visual_product_description", "recall_description", "remedies"]
label_fields = ["product_classification", "hazard_classification", "remedies_classification"]
source_fields = ["recall_number", "recall_date", "product_name", "product_quantity",
                 *text_fields[1:], *label_fields]
placeholders = {"", "-", "n/a", "na", "none", "null", "unknown", "not specified",
                "not available", "no description available", "a visual description is not available"}


def clean_text(value):
    if isinstance(value, str):
        value = " ".join(value.split())
        if value.casefold().rstrip(".") not in placeholders:
            return value
    return None


def empty_value(value):
    if isinstance(value, str):
        return clean_text(value) is None
    if isinstance(value, (list, dict)):
        return not value
    return pd.isna(value)


if DATA_PATH is None:
    candidates = [Path("dataset.json"), Path("dataset(1).json"), Path.home() / "Downloads" / "dataset.json"]
    DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("请在 DATA_PATH 指定原始 dataset.json。")
source_bytes = Path(DATA_PATH).expanduser().read_bytes()
records = json.loads(source_bytes)
if not isinstance(records, list) or not records or not all(isinstance(row, dict) for row in records):
    raise ValueError("原始 JSON 应为非空的召回记录列表。")
source = pd.DataFrame.from_records(records).reindex(columns=source_fields)
source.insert(0, "source_row", range(1, len(source) + 1))
dates = pd.to_datetime(source["recall_date"], format="ISO8601", errors="coerce", utc=True)
ids = pd.to_numeric(source["recall_number"], errors="coerce")
valid_id = (ids.gt(0) & ids.mod(1).eq(0)
            & ~source["recall_number"].map(lambda number: isinstance(number, bool)))

first_row_for_value = {}
first_rows = [first_row_for_value.setdefault(json.dumps(row, sort_keys=True), number)
              for number, row in enumerate(records, start=1)]
duplicate = source["source_row"].ne(first_rows)
print(f"{len(source):,} 行 · {ids.where(valid_id).nunique():,} 个召回编号 · "
      f"{int(duplicate.sum())} 行完全重复")
print(f"日期范围 {dates.min().date()} 至 {dates.max().date()}")
print(f"原始数据 SHA-256 {sha256(source_bytes).hexdigest()}")
field_overview = pd.DataFrame({
    "字段": source_fields,
    "缺失或占位行": [int(source[field].map(empty_value).sum()) for field in source_fields],
})
display(field_overview)

6,874 行 · 6,464 个召回编号 · 0 行完全重复
日期范围 2000-01-11 至 2024-12-26
原始数据 SHA-256 cf5b2bc18040aa4ec2cef6006fea858d6c267e07e0746686d3493d88548f1761


,字段,缺失或占位行
0,recall_number,0
1,recall_date,0
2,product_name,0
3,product_quantity,0
4,visual_product_description,213
5,recall_description,0
6,remedies,7
7,product_classification,0
8,hazard_classification,0
9,remedies_classification,0


## 2　基础清洗

清理明确的文本占位和空白，统一日期与已确认的标签拼写。完全相同的记录保留第一行，同一召回编号下的其他来源行保留各自的类别和文本。标签缺失或同一编号下的类别差异进入复核清单。数量原值保留，零值、缺失及非法值单独标为未知。产品外观、召回描述和处理措施原文继续用于后续案例核对。

In [2]:
allowed = {
    "product_classification": {
        "BABY_PRODUCTS", "CLOTHING_ACCESSORIES", "ELECTRICAL", "FURNITURE", "HOME_APPLIANCES",
        "OTHER", "OUTDOOR_EQUIPMENT", "SPORTS_RECREATION", "TOOLS_HARDWARE", "TOYS_CHILDREN", "TRANSPORTATION",
    },
    "hazard_classification": {
        "BURN", "CHOKING", "CRASH", "CUTS", "FALLING", "FIRE", "HEAT_RELATED_EXPLOSION",
        "IMPACT", "OTHER", "POISONING", "SAFETY_EQUIPMENT_MALFUNCTION",
    },
    "remedies_classification": {
        "DISPOSE", "NEW_INSTRUCTIONS", "REFUND", "REMEDY_NO_LONGER_AVAILABLE", "REPAIR", "REPLACE",
    },
}
hazard_spelling = {"HEAT_RELATED_EXPOLOSION": "HEAT_RELATED_EXPLOSION"}
clean = source.copy()
review_entries = []


def log_issue(mask, field, issue, decision, evidence=None):
    mask = mask.fillna(False)
    selected = source.loc[mask, ["source_row", "recall_number"]].copy()
    if selected.empty:
        return
    selected["field"] = field
    selected["issue"] = issue
    selected["decision"] = decision
    selected["original_value"] = source.loc[mask, field].map(repr)
    if evidence is not None:
        selected["evidence"] = evidence.loc[selected.index]
    review_entries.append(selected)


duplicate_origin = pd.Series(first_rows, index=source.index).map(lambda row: f"首行 {row}")
log_issue(duplicate, "recall_number", "完全重复行", "保留首行", duplicate_origin)
clean["recall_number"] = ids.where(valid_id).astype("Int64")
clean["recall_date"] = dates.dt.strftime("%Y-%m-%d")
clean["year"] = dates.dt.year.astype("Int64")
date_conflict = clean.groupby("recall_number")["recall_date"].transform("nunique").gt(1)
in_period = clean["year"].between(FIRST_YEAR, LAST_YEAR).fillna(False)
valid_period = valid_id & in_period & ~date_conflict
clean["split"] = "review"
clean.loc[valid_period & clean["year"].le(TRAIN_END), "split"] = "history"
clean.loc[valid_period & clean["year"].gt(TRAIN_END), "split"] = "evaluation"
log_issue(~valid_id, "recall_number", "编号缺失或非法", "保留来源行，暂不进入实验")
log_issue(~in_period, "recall_date", "日期非法、缺失或超出研究期", "保留来源行，暂不进入实验")
log_issue(date_conflict, "recall_date", "同一编号日期冲突", "核对公告后再按时间分期")

for field in text_fields:
    clean[field] = source[field].map(clean_text)
    log_issue(clean[field].isna(), field, "文本缺失、占位或格式异常", "记为未知，保留来源行")

for field in label_fields:
    aliases = hazard_spelling if field == "hazard_classification" else {}
    parsed = source[field].map(lambda value: [s.strip() for s in value if isinstance(s, str)]
                               if isinstance(value, list) else [])
    canonical = parsed.map(lambda labels: [aliases.get(label, label) for label in labels])
    clean[field] = canonical.map(lambda labels: list(dict.fromkeys(label for label in labels
                                                                   if label in allowed[field])))
    label_list_ok = source[field].map(lambda value: isinstance(value, list) and bool(value)
                                     and all(isinstance(label, str) for label in value))
    bad_label = ~label_list_ok | canonical.map(lambda labels: any(label not in allowed[field] for label in labels))
    conflict = clean[field].map(lambda labels: tuple(sorted(labels))).groupby(
        clean["recall_number"]
    ).transform("nunique").gt(1)
    log_issue(bad_label, field, "标签缺失或无法识别", "保留有效标签，异常类别待复核")
    log_issue(parsed.map(lambda labels: any(label in aliases for label in labels)),
              field, "已知拼写异体", "统一拼写", clean[field].map(repr))
    log_issue(conflict, field, "同一编号的标签组合不同", "保留来源行，按公告复核")
    if field == "product_classification":
        multiple = clean[field].map(len).gt(1)
        clean["category_needs_review"] = bad_label | conflict | multiple
        log_issue(multiple, field, "同一来源行有多个产品标签", "保留多标签，按公告复核")

quantity = pd.to_numeric(source["product_quantity"], errors="coerce")
clean["quantity_unknown"] = (~(quantity.gt(0) & quantity.lt(float("inf")))
                            | source["product_quantity"].map(lambda number: isinstance(number, bool)))
log_issue(clean["quantity_unknown"], "product_quantity", "零值、缺失或非法数量", "保留原值并标记未知")
clean = clean.loc[~duplicate].reset_index(drop=True)
review_columns = ["source_row", "recall_number", "field", "issue", "decision", "original_value", "evidence"]
quality_log = (pd.concat(review_entries, ignore_index=True).reindex(columns=review_columns)
               if review_entries else pd.DataFrame(columns=review_columns))
display(quality_log.groupby(["field", "issue"], sort=False).agg(
    来源行=("source_row", "nunique"), 召回编号=("recall_number", "nunique"),
).reset_index())
print(f"清洗后 {len(clean):,} 条来源记录。")

清洗后 6,874 条来源记录。


,field,issue,来源行,召回编号
0,visual_product_description,文本缺失、占位或格式异常,213,210
1,remedies,文本缺失、占位或格式异常,7,7
2,product_classification,同一编号的标签组合不同,174,42
3,hazard_classification,已知拼写异体,68,61
4,hazard_classification,同一编号的标签组合不同,200,63
5,remedies_classification,同一编号的标签组合不同,215,67
6,product_quantity,零值、缺失或非法数量,878,860


## 3　生成交接特征、导出与核对

用下方单元中的关键词和正则规则逐来源行生成召回年份、家电子类、失效方式和可见结构。以 2000–2018 年为历史期、2019–2024 年为评价期，核对召回编号及「编号 × 大类」组合数。默认导出到 preprocessing_output。合作实验可通过 DATA_PATH 读取导出的 project_input_records.json。

[预处理总览](preprocessing_overview.md) · [合作实验](全量召回数据_实验AB中文报告.ipynb) · [后续改进](improvement.md)

In [ ]:
import re

# 四个交接字段的词典、提取和校验规则。
HANDOFF_COLUMNS = ("recall_year", "appliance_subtype", "failure_modes", "visible_structures")

MODES = {
    '过热': r'overheat(?:s|ed|ing)?',
    '电气短路或电弧': r'short[- ]circuit(?:s|ed|ing)?|(?:electrical|electric) shorts?|arc(?:ing|s)',
    '破裂或断裂': r'break(?:s|ing)?|broken|crack(?:s|ed|ing)?|shatter(?:s|ed|ing)?|fracture(?:s|d)?',
    '松脱或分离': r'detach(?:es|ed|ing)?|separat(?:e|es|ed|ing)|(?:come|comes|become|becomes) loose|loosen(?:s|ed|ing)?|fall off',
    '泄漏': r'leak(?:s|ed|ing|age)?',
    '意外启动': r'(?:turn on|activate|start) (?:unexpectedly|unintentionally|by itself)|(?:unintentionally|unexpectedly) (?:turn on|activate|start)',
}


MODES['过热'] += r'|(?:get|gets|become) too hot|temperatures? (?:exceeds? allowable|can get too high)|elevated internal temperature|temperature of the storage drawer can become extremely high|heat to build up'


MODES['电气短路或电弧'] += r'|(?:electric|electrical) arc|short(?:s|ed)? out'


MODES['破裂或断裂'] += r'|ruptur(?:e|es|ed)|burst(?:s)?'


MODES['松脱或分离'] += r'|pulled loose|dislodg(?:e|es|ed)|detachments?|disengage|fall from (?:its housing|the ceiling|the wall)'


MODES['泄漏'] += r'|(?:flue gas|gas) (?:can|to) escape|gas to be released'


MODES['意外启动'] += r'|turn on (?:by themselves|without user input|spontaneously)|activated by accidental contact'

def matches(dictionary, phrase):
    return [name for name, pattern in dictionary.items()
            if re.search(r'\b(?:' + pattern + r')\b', phrase, re.I)]

TYPES = {
    '热水器': r'water heaters?',
    '锅炉': r'boilers?',
    '壁炉': r'fireplaces?',
    '暖炉': r'furnaces?',
    '取暖器': r'heaters?',
    '空调及热泵': r'air conditioners?|heat pumps?',
    '除湿机': r'dehumidifiers?',
    '冰箱及冷柜': r'refrigerators?|freezers?',
    '洗碗机': r'dishwashers?',
    '洗衣机': r'washing machines?|clothes washers?',
    '咖啡及浓缩咖啡设备': r'coffee\s?(?:makers?|brewers?|machines?|centers?)|espresso (?:makers?|machines?)',
    '压力锅': r'pressure cookers?',
    '灶具及烤箱': r'ranges?|ovens?|cooktops?',
    '风扇': r'fans?',
    '吸尘器': r'vacuums?',
}

EXCLUDE = re.compile(
    r'\b(?:range hoods?|oven mitts?|oven gloves?|vacuum flasks?|dutch ovens?|oven liners?|'
    r'oven rack guards?|cooktop fire suppressors?|drip pans?|fire suppression devices?|oven cleaner|'
    r'gas control valves for|water heater (?:gas )?temperature controls?|spas and hot tubs|'
    r'fireplace screens?|fireplace gas valves?|gas valves used in|remote control kits?|'
    r'attachment tools?|fan electric motor|coil drain pans?|boiler (?:operating|control|controller)|'
    r'control units for boilers|service kit|accessory (?:electric )?heaters?|defrost heaters?|'
    r'(?:spa|pool|immersion) heaters?)\b', re.I)

STRUCTURES = {
    "折叠或铰链": r"foldable|folding|fold[- ]lock|fold release|hinges?|hinged",
    "轮或脚轮": r"wheels|casters?|castors?",
    "束带或安全带": r"straps?|harness(?:es)?|seat belts?",
    "外部电源线或插头": r"power cords?|power cables?|electrical plugs?|power plugs?",
    "电池仓或其盖板": r"battery compartments?|battery covers?|battery doors?",
    "拉绳或珠链": r"drawstrings?|pull cords?|bead(?:ed)? chains?",
    "门或可开启盖": r"doors?|lids?|hinged covers?",
    "通风孔或格栅": r"air vents?|ventilation slots?|grilles?|grills?",
    "外露把手": r"handles?|handgrips?|grip bars?",
}


FEATURE_PATTERNS = {name: re.compile(rf"\b(?:{terms})\b", re.I) for name, terms in STRUCTURES.items()}


OUTCOME_WORDS = re.compile(
    r"\b(?:hazards?|recalled?|safety recall|risks? of|poses?|unsafe|danger(?:ous)?|"
    r"overheat\w*|chok\w*|strangl\w*|poison\w*|injur\w*|lacerat\w*|electrocut\w*|"
    r"amputat\w*|caught in|finger entrapment|fire hazard|burn hazard)\b", re.I,
)


def visible_structure(text):
    sentences = [s for s in re.split(r"[.!?;](?=\s|$)", text) if not OUTCOME_WORDS.search(s)]
    return {name for name, pattern in FEATURE_PATTERNS.items() if any(pattern.search(s) for s in sentences)}

def recall_year(date):
    if isinstance(date, str) and len(date) >= 4 and date[:4].isdigit():
        return int(date[:4])
    return None


def extract_row_features(row):
    """Generate the four features on the same source row as the texts and labels."""
    product_name = row.get("product_name") or ""
    subtype = None
    if "HOME_APPLIANCES" in row["product_classification"] and not EXCLUDE.search(product_name):
        subtype = next(iter(matches(TYPES, product_name)), None)
    return {
        "recall_year": recall_year(row.get("recall_date")),
        "appliance_subtype": subtype,
        "failure_modes": matches(MODES, row.get("recall_description") or ""),
        "visible_structures": sorted(visible_structure(row.get("visual_product_description") or "")),
    }


def validate_handoff(rows):
    """Check that every exported record has valid handoff features."""
    fields = set(HANDOFF_COLUMNS)
    for row in rows:
        number = row.get("recall_number")
        present = fields.intersection(row)
        if present != fields:
            raise ValueError(f"召回 {number} 缺少交接字段 {sorted(fields - present)}，请重新运行预处理。")

        year = row["recall_year"]
        if type(year) is not int or year != recall_year(row.get("recall_date")):
            raise ValueError(f"召回 {number} 的年份和日期不一致。")
        subtype = row["appliance_subtype"]
        if subtype is not None and (
            not isinstance(subtype, str) or subtype not in TYPES
            or "HOME_APPLIANCES" not in row["product_classification"]
        ):
            raise ValueError(f"召回 {number} 的家电子类不在交接词典或大类范围内。")
        for field, allowed in (("failure_modes", MODES), ("visible_structures", STRUCTURES)):
            values = row[field]
            if not isinstance(values, list) or not all(isinstance(v, str) and v in allowed for v in values):
                raise ValueError(f"召回 {number} 的 {field} 与交接词典不符。")

In [4]:
features = pd.DataFrame(
    [extract_row_features(row) for row in clean.to_dict("records")], index=clean.index,
)
clean[list(HANDOFF_COLUMNS)] = features
clean["recall_year"] = clean["recall_year"].astype("Int64")
assert clean["recall_year"].equals(clean["year"])

eligible = clean["split"].isin(["history", "evaluation"]) & (
    clean["product_classification"].map(bool) & clean["hazard_classification"].map(bool)
)
project_records = clean.loc[
    eligible, ["source_row", *source_fields, *HANDOFF_COLUMNS, "split", "category_needs_review"],
].copy()
project_records[text_fields] = project_records[text_fields].fillna("")
project_records["product_quantity"] = pd.to_numeric(project_records["product_quantity"], errors="coerce").where(
    ~clean.loc[eligible, "quantity_unknown"]
).astype(float)

# 交接数据沿用合作 Notebook 的危险标签拼写。
project_spelling = {corrected: original for original, corrected in hazard_spelling.items()}
project_records["hazard_classification"] = project_records["hazard_classification"].map(
    lambda labels: [project_spelling.get(label, label) for label in labels]
)
coverage = (project_records[["recall_number", "product_classification", "split"]]
            .explode("product_classification").drop_duplicates())
display(coverage.groupby("split", sort=False).agg(
    召回编号=("recall_number", "nunique"), 编号与类别组合=("recall_number", "size"),
))

validate_handoff(project_records.to_dict("records"))
assert clean["source_row"].is_unique
assert len(clean) + int(duplicate.sum()) == len(source)
assert project_records.groupby("recall_number")["split"].nunique().le(1).all()
print("已核对来源行、精确去重、时间分期和四列交接特征。")
display(project_records.loc[project_records["recall_number"].eq(7984), ["recall_number", *HANDOFF_COLUMNS]])

if EXPORT_DIR is not None:
    export = Path(EXPORT_DIR).expanduser()
    export.mkdir(parents=True, exist_ok=True)
    clean.to_json(export / "preprocessed_records.json", orient="records", force_ascii=False, indent=2)
    project_records.to_json(export / "project_input_records.json", orient="records", force_ascii=False, indent=2)
    quality_log.to_csv(export / "quality_review.csv", index=False)
    print(f"已导出清洗表、项目输入及复核清单到 {export}。")

已核对来源行、精确去重、时间分期和四列交接特征。
已导出清洗表、项目输入及复核清单到 preprocessing_output。


,召回编号,编号与类别组合
split,,
history,4978,5016
evaluation,1486,1491


,recall_number,recall_year,appliance_subtype,failure_modes,visible_structures
2771,7984,2017,空调及热泵,[过热],[通风孔或格栅]


## 可选检查　松脱、窒息与处理措施

在 2019–2024 年的玩具及儿童用品记录中，只对该大类内有一条来源行、且召回描述和危险标签齐全的编号计数。分组比较有没有提到「松脱或分离」。先看窒息标签的比例，再在报有窒息且具有实际措施标签的案例中比较退款比例。「补救不可用」作为状态处理。

这个对照按公告中的文字和标签寻找相似案例。组间差异需要结合具体玩具类型和部件来解释。


In [5]:
toy_rows = clean.loc[
    clean["split"].eq("evaluation")
    & clean["product_classification"].map(lambda labels: "TOYS_CHILDREN" in labels)
]
toy_cases = toy_rows.loc[
    ~toy_rows["recall_number"].duplicated(keep=False)
    & toy_rows["recall_description"].notna()
    & toy_rows["hazard_classification"].map(bool)
]
mentions_detachment = toy_cases["failure_modes"].map(lambda modes: "松脱或分离" in modes)


def fraction_text(count, total):
    return f"{count}/{total} ({count / total:.1%})" if total else "—"


comparison = []
for mentioned, name in [(True, "提到松脱或分离"), (False, "未提到松脱或分离")]:
    cases = toy_cases.loc[mentions_detachment.eq(mentioned)]
    choking = cases.loc[cases["hazard_classification"].map(lambda tags: "CHOKING" in tags)]
    action_labels = choking["remedies_classification"].map(lambda tags: set(tags) - {"REMEDY_NO_LONGER_AVAILABLE"})
    with_action = action_labels.loc[action_labels.map(bool)]
    refunds = int(with_action.map(lambda tags: "REFUND" in tags).sum())
    comparison.append({
        "召回描述": name, "召回数": len(cases),
        "带窒息标签的比例": fraction_text(len(choking), len(cases)),
        "窒息样本中有措施标签": len(with_action),
        "其中带退款标签的比例": fraction_text(refunds, len(with_action)),
    })

retained = toy_cases["recall_number"].nunique()
available = toy_rows["recall_number"].nunique()
print(f"2019–2024 年玩具及儿童用品可用 {retained}/{available} 个召回编号。")
focused_check = pd.DataFrame(comparison)
display(focused_check)

2019–2024 年玩具及儿童用品可用 210/215 个召回编号。


,召回描述,召回数,带窒息标签的比例,窒息样本中有措施标签,其中带退款标签的比例
0,提到松脱或分离,66,55/66 (83.3%),54,40/54 (74.1%)
1,未提到松脱或分离,144,25/144 (17.4%),24,20/24 (83.3%)
